# Epsilon Fund - Walk-Forward Validation
Uses `infrastructure/walk_forward/` to run rolling Optuna optimisation and evaluate OOS robustness.

---

In [17]:
import sys
import os
import pandas as pd
import numpy as np


# ── Set your repo root path ────────────────────────────────────────────────────
ROOT = r'/Users/justiniturregui/Desktop/epsilon/github/Epsilon-Quant-Research'  # ← change to yours
# ──────────────────────────────────────────────────────────────────────────────

sys.path.append(ROOT + '/infrastructure/data')
sys.path.append(ROOT + '/infrastructure/walkforward')

from binance_client import get_binance_client, get_data
from wf_engine import walk_forward
from wf_visualizer import plot_walk_forward_results

---
## Data

**Pairs** — any Binance pair in `BASEQUOTE` format (e.g. `BTCUSDT`, `ETHUSDT`, `SOLUSDT`, `BNBUSDT`).  
Verify availability at [binance.com/en/trade](https://www.binance.com/en/trade).

**Intervals** — `'1m'` `'5m'` `'15m'` `'1h'` `'4h'` `'1d'` `'1w'`

**Lookback** — days of history: `365` (1y) · `730` (2y) · `1825` (5y) · `2555` (7y, recommended minimum)

In [ ]:
SYMBOL   = 'BTCUSDT'
INTERVAL = '1d'
LOOKBACK = 2555  # must be >= (train_bars + test_bars) * n_folds desired


# ── Multiple pairs (optional) ──────────────────────────────────────────────────
# SYMBOLS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']
# data_dict = get_multiple_data(client, SYMBOLS, INTERVAL, LOOKBACK)
# Access via: data_dict['BTCUSDT_1d'], data_dict['ETHUSDT_1d'] ...
# ──────────────────────────────────────────────────────────────────────────────

client   = get_binance_client()
df = get_data(client, 'BTCUSDT', '1d', LOOKBACK)
print(f'Data: {df.index[0].date()} → {df.index[-1].date()}  ({len(df)} bars)')
df.tail(3)

Data: 2019-03-29 → 2026-03-26  (2555 bars)


,Open,High,Low,Close,Volume
Time,,,,,
2026-03-24,70906.45,71400.00,68923.07,70556.74,20702.42244
2026-03-25,70556.74,72026.09,70408.00,71336.53,17719.50066
2026-03-26,71336.53,71436.82,68632.40,68909.06,13912.68700


---
## Parameter Configuration

Define which parameters to optimise and anchor

In [ ]:

# ── parameter search space ────────────────────────────────────────────────────
# Format: 'param_name': ('int' | 'float', lo, hi)
PARAM_DEFS = {
    'ema_span':          ('int',   5,    40),
    'swing_caution':     ('int',   3,    14),
    'swing_stop':        ('int',   3,    10),
    'atr_caution':       ('int',   10,   30),
    'atr_stop':          ('int',   10,   30),
    'atr_size':          ('int',   3,    14),
    'caution_threshold': ('float', 0.8,  2.5),
    'adx_override':      ('int',   40,   80),
    'stop_atr_scale':    ('float', 0.5,  2.0),
    'risk_per_trade':    ('float', 0.005, 0.05),
    'max_leverage':      ('float', 1.0,  3.0),
    'stop_mult_pos_both':    ('float', 0.5, 2.5),
    'stop_mult_pos_caution': ('float', 0.1, 0.9),
    'stop_mult_pos_short':   ('float', 0.1, 1.5),
    'stop_mult_pos_normal':  ('float', 0.8, 2.0),
    'stop_mult_ent_both':    ('float', 0.5, 2.5),
    'stop_mult_ent_caution': ('float', 0.1, 0.9),
    'stop_mult_ent_short':   ('float', 0.1, 1.5),
    'stop_mult_ent_normal':  ('float', 0.5, 1.5),
}

# ── anchored params (set after a stability run; empty first time) ─────────────
# These bypass Optuna and are held constant across all folds.
# Populate using stability_df results: fix params with CV < 0.15
FIXED_PARAMS = {
    # 'ema_span': 20,
}

---
## Strategy

**Available columns:** `Open` `High` `Low` `Close` `Volume`

**Required output:** a `position` column — `1` long · `0` flat · `-1` short  
**Optional:** `position_size` column (0–1) to use fractional capital

> Signals are shifted 1 bar by the engine — no need to shift manually.

Wrap your notebook strategy into a function that takes `(df_slice, params)` and returns a strategy dataframe.
This is the only cell that changes between strategies.

In [ ]:
def my_strategy(df_slice: pd.DataFrame, params: dict) -> pd.DataFrame:
    """
    Compute indicators, generate signals, build position/position_size columns.
    Must return a DataFrame with DatetimeIndex and at minimum:
        'Close'         — price
        'position'      — 1 / 0 / -1
        'position_size' — leveraged sizing
    """
    df = df_slice.copy()

    # ── unpack params ─────────────────────────────────────────────────────────
    EMA_SPAN          = params['ema_span']
    SWING_CAUTION     = params['swing_caution']
    SWING_STOP        = params['swing_stop']
    ATR_CAUTION       = params['atr_caution']
    ATR_STOP          = params['atr_stop']
    ATR_SIZE          = params['atr_size']
    CAUTION_THRESHOLD = params['caution_threshold']
    ADX_OVERRIDE      = params['adx_override']
    STOP_ATR_SCALE    = params['stop_atr_scale']
    RISK_PER_TRADE    = params['risk_per_trade']
    MAX_LEVERAGE      = params['max_leverage']

    STOP_MULT_POS_BOTH    = params['stop_mult_pos_both']
    STOP_MULT_POS_CAUTION = params['stop_mult_pos_caution']
    STOP_MULT_POS_SHORT   = params['stop_mult_pos_short']
    STOP_MULT_POS_NORMAL  = params['stop_mult_pos_normal']
    STOP_MULT_ENT_BOTH    = params['stop_mult_ent_both']
    STOP_MULT_ENT_CAUTION = params['stop_mult_ent_caution']
    STOP_MULT_ENT_SHORT   = params['stop_mult_ent_short']
    STOP_MULT_ENT_NORMAL  = params['stop_mult_ent_normal']

    # ── indicators ────────────────────────────────────────────────────────────
    df['EMA']          = df['Close'].ewm(span=EMA_SPAN, adjust=False).mean()

    df['Swing_Hi_Cau'] = df['High'].rolling(SWING_CAUTION).max()
    df['Swing_Lo_Cau'] = df['Low'].rolling(SWING_CAUTION).min()
    df['Swing_Hi_Stp'] = df['High'].rolling(SWING_STOP).max()

    # ATR helper
    def atr(period):
        hl = df['High'] - df['Low']
        hc = (df['High'] - df['Close'].shift(1)).abs()
        lc = (df['Low']  - df['Close'].shift(1)).abs()
        return pd.concat([hl, hc, lc], axis=1).max(axis=1).ewm(span=period, adjust=False).mean()

    df['ATR_Cau'] = atr(ATR_CAUTION)
    df['ATR_Stp'] = atr(ATR_STOP)
    df['ATR_Sz']  = atr(ATR_SIZE)

    # ADX
    period = 14
    up   = df['High'].diff();  down = -df['Low'].diff()
    pdm  = up.where((up > down) & (up > 0), 0.0)
    ndm  = down.where((down > up) & (down > 0), 0.0)
    atr14 = atr(period)
    pdi  = 100 * pdm.ewm(span=period, adjust=False).mean() / atr14
    ndi  = 100 * ndm.ewm(span=period, adjust=False).mean() / atr14
    dx   = (100 * (pdi - ndi).abs() / (pdi + ndi).replace(0, np.nan)).fillna(0)
    df['ADX_14'] = dx.ewm(span=period, adjust=False).mean()

    # 1-bar forward shift (lookahead bias prevention)
    for col in ['EMA', 'Swing_Hi_Cau', 'Swing_Lo_Cau', 'Swing_Hi_Stp',
                'ATR_Cau', 'ATR_Stp', 'ATR_Sz', 'ADX_14']:
        df[col] = df[col].shift(1)

    # ── caution & entry flags ──────────────────────────────────────────────────
    df['Caution_Long']  = (df['Swing_Hi_Cau'] - df['Low'])  > CAUTION_THRESHOLD * df['ATR_Cau']
    df['Caution_Short'] = ((df['High'] - df['Swing_Lo_Cau']) > CAUTION_THRESHOLD * df['ATR_Cau']) | (df['Close'] > df['EMA'])
    df['Entry_Long']    = (df['Close'] > df['EMA']) & (~df['Caution_Long'] | (df['ADX_14'] > ADX_OVERRIDE))

    # ── position sizing ────────────────────────────────────────────────────────
    df['position_size_raw'] = (RISK_PER_TRADE / (df['ATR_Sz'] / df['Close'])).clip(0.1, MAX_LEVERAGE)

    # ── execution loop ────────────────────────────────────────────────────────
    n = len(df)
    position      = [0] * n
    position_size = [0.0] * n
    stop_arr      = [np.nan] * n

    in_position   = 0
    stop_loss     = np.nan
    current_size  = 0.0

    for i in range(1, n):
        prev = df.iloc[i - 1]
        curr = df.iloc[i]

        if in_position == 0:
            if prev['Entry_Long']:
                in_position  = 1
                current_size = curr['position_size_raw']

                # entry-day stop
                cl = prev['Caution_Long']; cs = prev['Caution_Short']
                if   cl and cs: sm = STOP_MULT_ENT_BOTH
                elif cl:        sm = STOP_MULT_ENT_CAUTION
                elif cs:        sm = STOP_MULT_ENT_SHORT
                else:           sm = STOP_MULT_ENT_NORMAL

                stop_distance = curr['ATR_Stp'] * sm * STOP_ATR_SCALE
                stop_loss     = (curr['Swing_Hi_Stp'] - stop_distance)

        else:
            # exit check
            if prev['Close'] < stop_loss:
                in_position  = 0
                current_size = 0.0
                stop_loss    = np.nan
            else:
                # trail stop
                cl = curr['Caution_Long']; cs = curr['Caution_Short']
                if   cl and cs: sm = STOP_MULT_POS_BOTH
                elif cl:        sm = STOP_MULT_POS_CAUTION
                elif cs:        sm = STOP_MULT_POS_SHORT
                else:           sm = STOP_MULT_POS_NORMAL

                stop_distance = curr['ATR_Stp'] * sm * STOP_ATR_SCALE
                stop_loss     = max(stop_loss, curr['Swing_Hi_Stp'] - stop_distance)

        position[i]      = in_position
        position_size[i] = current_size
        stop_arr[i]      = stop_loss

    df['position']      = position
    df['position_size'] = position_size
    df['stop_loss']     = stop_arr

    # ── clean NaN rows from indicator warmup ──────────────────────────────────
    indicator_cols = ['EMA', 'ATR_Cau', 'ADX_14', 'Swing_Hi_Cau']
    df.dropna(subset=indicator_cols, inplace=True)
    df['position']      = df['position'].fillna(0).astype(int)
    df['position_size'] = df['position_size'].fillna(0.0)

    return df, indicator_cols


---
## IS Sanity Check

**CRUCIAL:** Using backtesting function to make sure strategy runs well before walk-forward testing. Saves time from Optuna runs.

---
## Run Walk-Forward
Simulates how the strategy would have performed if re-optimised periodically
in live trading, and exposes whether good IS performance survives unseen data.

**IS/OOS:** `TRAIN_BARS` `TEST_BARS` rolling training and testing windows, Optuna optimises free parameters on each training windows independently, then evaluates the best params OOS on the following test window (aim for 2:1 or 3:1 ratio minimum)

**Folds Setup** - trials and parameter anchoring: `N_TRIALS` `FIXED_PARAMS` (choose parameters with CV < 0.15 from prior stability run to reduce search space, improve OOS credibility)


In [ ]:
# ── walk-forward windows ──────────────────────────────────────────────────────
TRAIN_BARS  = 730   # ~2 years training
TEST_BARS   = 365   # ~1 year test per fold  (3:1 IS:OOS ratio)
BURNIN_BARS = 60    # bars prepended to test for indicator warmup (no loss of efficiency from folds)
N_TRIALS    = 400   # Optuna trials per fold - more better optimise but slower (300 - 500 for daily)
COST        = 0.001 # round-trip cost fraction

results = walk_forward(
    df           = df,
    strategy_fn  = my_strategy,
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    train_bars   = TRAIN_BARS,
    test_bars    = TEST_BARS,
    burnin_bars  = BURNIN_BARS,
    n_trials     = N_TRIALS,
    cost         = COST,
    save_csv     = None,
)

Walk-forward: 5 fold(s)  train=730  test=365  burnin=60  trials=400
  Fold 1: train 2019-03-29 → 2021-03-27  | test 2021-03-28 → 2022-03-27
  Fold 2: train 2020-03-28 → 2022-03-27  | test 2022-03-28 → 2023-03-27
  Fold 3: train 2021-03-28 → 2023-03-27  | test 2023-03-28 → 2024-03-26
  Fold 4: train 2022-03-28 → 2024-03-26  | test 2024-03-27 → 2025-03-26
  Fold 5: train 2023-03-28 → 2025-03-26  | test 2025-03-27 → 2026-03-26

────────────────────────────────────────────────────────────
Fold 1/5  train: 2019-03-29 → 2021-03-27  test: 2021-03-28 → 2022-03-27


  0%|          | 0/400 [00:00<?, ?it/s]


  IS  → Sharpe: 1.99  Return: 1087.10%  DD: -65.31%  Calmar: 16.65  Trades: 0
  OOS → Sharpe: 0.03  Return: -2.52%  DD: -21.52%  Calmar: -0.12  Trades: 0

  Best params: {'ema_span': 18, 'swing_caution': 14, 'swing_stop': 8, 'atr_caution': 22, 'atr_stop': 13, 'atr_size': 4, 'caution_threshold': 0.8987421406859392, 'adx_override': 75, 'stop_atr_scale': 1.4016725176148133, 'risk_per_trade': 0.03686326600082205, 'max_leverage': 1.041168988591605, 'stop_mult_pos_both': 2.4398197043239884, 'stop_mult_pos_caution': 0.7659541126403374, 'stop_mult_pos_short': 0.3972747549495866, 'stop_mult_pos_normal': 1.0181899606485207, 'stop_mult_ent_both': 0.8668090197068676, 'stop_mult_ent_caution': 0.3433937943676302, 'stop_mult_ent_short': 0.834659004285133, 'stop_mult_ent_normal': 0.9319450186421158}

────────────────────────────────────────────────────────────
Fold 2/5  train: 2020-03-28 → 2022-03-27  test: 2022-03-28 → 2023-03-27


  0%|          | 0/400 [00:00<?, ?it/s]


  IS  → Sharpe: 2.56  Return: 1090.36%  DD: -20.90%  Calmar: 52.18  Trades: 46
  OOS → Sharpe: -0.32  Return: -30.55%  DD: -61.05%  Calmar: -0.50  Trades: 0

  Best params: {'ema_span': 29, 'swing_caution': 14, 'swing_stop': 3, 'atr_caution': 29, 'atr_stop': 11, 'atr_size': 3, 'caution_threshold': 2.123716515441338, 'adx_override': 70, 'stop_atr_scale': 1.3445371134996007, 'risk_per_trade': 0.04579674644572064, 'max_leverage': 2.576266423201014, 'stop_mult_pos_both': 2.201609245287791, 'stop_mult_pos_caution': 0.6892961490945857, 'stop_mult_pos_short': 0.8364065357817274, 'stop_mult_pos_normal': 1.7859224808388898, 'stop_mult_ent_both': 2.0439124184212147, 'stop_mult_ent_caution': 0.8904551299655128, 'stop_mult_ent_short': 1.4398934984001452, 'stop_mult_ent_normal': 1.0944078703765996}

────────────────────────────────────────────────────────────
Fold 3/5  train: 2021-03-28 → 2023-03-27  test: 2023-03-28 → 2024-03-26


  0%|          | 0/400 [00:00<?, ?it/s]

[W 2026-03-26 17:25:42,973] Trial 144 failed with parameters: {'ema_span': 13, 'swing_caution': 9, 'swing_stop': 8, 'atr_caution': 18, 'atr_stop': 16, 'atr_size': 10, 'caution_threshold': 1.8582504617471338, 'adx_override': 75, 'stop_atr_scale': 1.1165598109811286, 'risk_per_trade': 0.03225309652080808, 'max_leverage': 1.5618266834752514, 'stop_mult_pos_both': 0.7435762637691951, 'stop_mult_pos_caution': 0.35785876455304294, 'stop_mult_pos_short': 0.22068578827149662, 'stop_mult_pos_normal': 1.3131863834304098, 'stop_mult_ent_both': 2.452991278462826, 'stop_mult_ent_caution': 0.6325912256880448, 'stop_mult_ent_short': 1.1622332075583335, 'stop_mult_ent_normal': 0.9913828653728283} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/Users/justiniturregui/Desktop/epsilon/g

--- Logging error ---
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/optuna/progress_bar.py", line 24, in emit
    tqdm.write(msg)
    ~~~~~~~~~~^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/std.py", line 720, in write
    with cls.external_write_mode(file=file, nolock=nolock):
         ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/contextlib.py", line 148, in __exit__
    next(self.gen)
    ~~~~^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/std.py", line 750, in external_write_mode
    inst.refresh(nolock=True)
    ~~~~~~~~~~~~^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/std.py", line 1347, in refresh
    self.display()
    ~~~~~~~~~~~~^^
  File "/Library/Framework

KeyboardInterrupt: 

## 5 · Results

In [ ]:
# ── per-fold table ────────────────────────────────────────────────────────────
display_cols = [
    'fold', 'train_start', 'train_end', 'test_start', 'test_end',
    'train_sharpe', 'train_return', 'train_drawdown',
    'test_sharpe',  'test_return',  'test_drawdown',
    'test_trades',  'test_winrate', 'optuna_score',
]
results['results_df'][display_cols]

,fold,train_start,train_end,test_start,test_end,train_sharpe,train_return,train_drawdown,test_sharpe,test_return,test_drawdown,test_trades,test_winrate,optuna_score
0,1,2019-03-29,2021-03-27,2021-03-28,2022-03-27,1.988781,10.871010,-0.653091,0.033118,-0.025212,-0.215169,0,0.000000,-999.000000
1,2,2020-03-28,2022-03-27,2022-03-28,2023-03-27,2.563584,10.903581,-0.208977,-0.321601,-0.305510,-0.610519,0,0.000000,0.868993
2,3,2021-03-28,2023-03-27,2023-03-28,2024-03-26,-0.226292,-0.534694,-0.763966,2.229984,1.394705,-0.193792,0,0.000000,-999.000000
3,4,2022-03-28,2024-03-26,2024-03-27,2025-03-26,0.724191,0.643884,-0.671253,0.754881,0.321120,-0.348612,0,0.000000,-999.000000
4,5,2023-03-28,2025-03-26,2025-03-27,2026-03-26,1.434790,1.435023,-0.197701,0.124217,0.007095,-0.186149,13,0.461538,-999.000000


In [ ]:
# ── parameter stability ───────────────────────────────────────────────────────
# Params with CV < 0.15 are good candidates to add to FIXED_PARAMS for the next run.
results['stability_df'].sort_values('cv')

,param,median,std,cv,fixed,stable
18,stop_mult_ent_normal,0.940531,0.099563,0.105858,False,True
17,stop_mult_ent_short,1.222951,0.215113,0.175897,False,False
9,risk_per_trade,0.036863,0.006565,0.178094,False,False
7,adx_override,58.000000,10.353743,0.178513,False,False
4,atr_stop,16.000000,2.856571,0.178536,False,False
0,ema_span,33.000000,7.402702,0.224324,False,False
14,stop_mult_pos_normal,1.541953,0.349627,0.226743,False,False
12,stop_mult_pos_caution,0.689296,0.179233,0.260023,False,False
10,max_leverage,2.576266,0.676251,0.262493,False,False
8,stop_atr_scale,1.344537,0.381784,0.283952,False,False


In [ ]:
# ── consensus params (copy into your notebook strategy or grid_search.py) ─────
print('Consensus parameters:')
for k, v in results['consensus_params'].items():
    print(f'  {k:<30} = {v}')

Consensus parameters:
  ema_span                       = 33
  swing_caution                  = 10
  swing_stop                     = 5
  atr_caution                    = 22
  atr_stop                       = 16
  atr_size                       = 8
  caution_threshold              = 0.8987
  adx_override                   = 58
  stop_atr_scale                 = 1.3445
  risk_per_trade                 = 0.0369
  max_leverage                   = 2.5763
  stop_mult_pos_both             = 1.8002
  stop_mult_pos_caution          = 0.6893
  stop_mult_pos_short            = 0.7585
  stop_mult_pos_normal           = 1.542
  stop_mult_ent_both             = 1.1473
  stop_mult_ent_caution          = 0.8547
  stop_mult_ent_short            = 1.223
  stop_mult_ent_normal           = 0.9405


## 6 · Charts

In [ ]:
plot_walk_forward_results(
    results      = results,
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    benchmark_data = df,          # BTC buy-and-hold comparison
    show           = True,
    save_html_dir  = 'outputs/',  # set None to skip saving
)

✓ Saved fold performance chart → outputs/wf_fold_performance.html


ValueError: Vertical spacing cannot be greater than (1 / (rows - 1)) = 0.111111.
The resulting plot would have 10 rows (rows=10).

## 7 · Full OOS Backtest (visual check)
Run `engine.backtest` directly on the combined OOS dataframe to get the standard chart.

In [ ]:
import sys
sys.path.append(os.path.join(ROOT, 'backtester'))
from engine import backtest

if results['oos_combined_df'] is not None:
    _ = backtest(
        data           = results['oos_combined_df'],
        cost           = COST,
        show_plot      = True,
        save_html      = 'outputs/wf_oos_full.html',
        show_trades    = True,
        benchmark_data = df,
    )

✓ Saved interactive chart to: outputs/wf_oos_full.html


✓ Saved trade chart to: outputs/wf_oos_full_trades.html


---
## 8 · Robustness Analysis

Three additional tests that go beyond basic walk-forward:

1. **Plateau analysis** — sweep each free parameter across its range while holding others at consensus values. Broad, flat score curves → robust; narrow spikes → fragile/overfit.
2. **Perturbation test** — randomly jitter ALL free parameters simultaneously by ±5%, ±10%, ±20% of their range. Measures whether the optimum is a broad hill (robust) or a sharp needle (overfit).
3. **Transaction cost stress test** — re-run the combined OOS backtest at 1×, 1.5×, 2×, 3× the base cost. Fragile strategies collapse; robust ones degrade gradually.

> **Key insight**: The stability table (CV across folds) tells you *"does the optimizer consistently pick the same value?"*  
> Plateau analysis tells you *"if that value were slightly wrong, would performance collapse?"*  
> You need both. A parameter can be stable (low CV) but fragile (steep cliff around the chosen value).

In [ ]:
from wf_engine import plateau_analysis, plateau_summary

# ── 1-D sensitivity sweeps around consensus params ─────────────────────────
# Uses the full training dataset to evaluate how sensitive the score is
# to each individual parameter.  Takes ~2-5 min depending on strategy.

sweep_results = plateau_analysis(
    df           = df,
    strategy_fn  = my_strategy,
    base_params  = results['consensus_params'],
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    cost         = COST,
    n_steps      = 20,         # points per parameter sweep
)

# ── verdicts: which params sit on a plateau vs a cliff ─────────────────────
verdict_df = plateau_summary(
    sweep_results,
    base_params = results['consensus_params'],
    threshold   = 0.20,        # 20% score drop still acceptable
)

In [ ]:
import matplotlib.pyplot as plt

# ── plot 1-D sweep curves ─────────────────────────────────────────────────
free_params = [k for k in PARAM_DEFS if k not in FIXED_PARAMS]
n_params = len(free_params)
ncols = 3
nrows = (n_params + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 3.5*nrows))
axes = axes.flatten()

for idx, name in enumerate(free_params):
    ax = axes[idx]
    sdf = sweep_results[name].dropna(subset=['score'])
    if len(sdf) == 0:
        ax.set_title(f'{name} (no valid data)')
        continue

    ax.plot(sdf['value'], sdf['score'], 'o-', markersize=3, linewidth=1.2)

    # mark the consensus value
    consensus_val = results['consensus_params'].get(name)
    if consensus_val is not None:
        ax.axvline(consensus_val, color='red', linestyle='--', alpha=0.6, label='consensus')

    # shade the plateau region (within 20% of peak)
    peak = sdf['score'].max()
    cutoff = peak * 0.80
    ax.axhline(cutoff, color='green', linestyle=':', alpha=0.4, label='80% of peak')
    ax.fill_between(sdf['value'], cutoff, sdf['score'],
                    where=sdf['score'] >= cutoff, alpha=0.15, color='green')

    ax.set_title(name, fontsize=9)
    ax.set_ylabel('score', fontsize=8)
    ax.tick_params(labelsize=7)

# hide unused subplots
for idx in range(n_params, len(axes)):
    axes[idx].set_visible(False)

fig.suptitle('Parameter Plateau Analysis — 1D Sweeps', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
from wf_engine import perturbation_test

# ── neighbourhood perturbation ────────────────────────────────────────────
# Randomly jitters ALL free params simultaneously.
# If mean score degrades >15% at ±10% offset, the optimum is fragile.

perturb_df = perturbation_test(
    df           = df,
    strategy_fn  = my_strategy,
    base_params  = results['consensus_params'],
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    cost         = COST,
    pct_offsets  = (0.05, 0.10, 0.20),   # ±5%, ±10%, ±20% of range
    n_samples    = 50,                     # random perturbations per offset level
)

In [ ]:
from wf_engine import cost_stress_test

# ── transaction cost stress test ──────────────────────────────────────────
# Robust strategies degrade gradually; fragile ones collapse at 2-3x cost.

if results['oos_combined_df'] is not None:
    cost_df = cost_stress_test(
        oos_combined_df  = results['oos_combined_df'],
        cost_multipliers = (1.0, 1.5, 2.0, 3.0),
        base_cost        = COST,
    )
else:
    print('No combined OOS dataframe — skip cost stress test')

---
### How to interpret these results

**Plateau analysis verdicts:**
- `robust plateau` (>60% of range within 20% of peak) → safe to optimise, unlikely to overfit
- `moderate` (30-60%) → the parameter matters, tighten its range around the plateau region
- `FRAGILE` (<30%) → the optimizer is latching onto a narrow spike; consider fixing this param at a sensible default or collapsing it into a simpler parameterization

**Perturbation test:**
- <5% degradation at ±10% offset → excellent robustness
- 5-15% degradation at ±10% → acceptable, typical for a reasonably parameterized strategy  
- \>15% degradation at ±10% → the optimum is fragile; reduce free parameters

**Cost stress test:**
- Sharpe still positive at 2× cost → tradeable
- Sharpe collapses at 1.5× cost → too sensitive to execution quality

---
### Next steps: reducing parameter count

The single most impactful change is **collapsing the 8 stop-multiplier params** into 2-3:

```python
# Instead of 8 separate stop multipliers, derive them from 2 base params:
# 'base_stop_mult' (0.5 - 2.5) — the normal/default multiplier
# 'caution_discount' (0.1 - 0.8) — how much to tighten when caution flags fire
#
# Derivation:
#   normal   = base_stop_mult
#   caution  = base_stop_mult * caution_discount
#   short    = base_stop_mult * (1 - caution_discount) / 2  (somewhere between)
#   both     = base_stop_mult * caution_discount * 0.8      (tightest)
#
# Apply same derivation for entry vs position stops (or use the same set for both,
# which would take you from 8 params → 2).
```

This takes the strategy from **18 free params → 12** (or even 10), which is a much better ratio against 400 Optuna trials and directly reduces overfitting risk.